# Project 5 — Credit Risk Analysis for Loan Approval

**Objective:** Analyze real loan applicant data to assess credit risk and build a Logistic Regression model to predict loan approval/default.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                              roc_auc_score, roc_curve, ConfusionMatrixDisplay)

## Step 1: Load Real Dataset

In [ ]:
df = pd.read_csv('5_loan_data.csv')   # ← Use the REAL dataset
print(f'Shape: {df.shape}')
df.head()

## Step 2: Data Preprocessing

In [ ]:
print('Missing values:')
print(df.isnull().sum())
df.dropna(inplace=True)

print('\nData types:')
print(df.dtypes)

In [ ]:
# Label encode any categorical columns
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
if cat_cols:
    le = LabelEncoder()
    for col in cat_cols:
        df[col] = le.fit_transform(df[col].astype(str))
    print(f'Label Encoded: {cat_cols}')
else:
    print('No categorical columns — dataset is fully numeric')

# Scale numerical features
target_col = 'loan_status'
X_raw = df.drop(columns=[target_col])
y = df[target_col]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
X = pd.DataFrame(X_scaled, columns=X_raw.columns)
print('✅ Features scaled with StandardScaler')

## Step 3: EDA — Target Distribution & Correlation Heatmap

In [ ]:
print('Loan Status Distribution:')
print(y.value_counts())
print(f'Default rate: {y.mean()*100:.1f}%')

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

labels = ['Approved (0)', 'Default (1)']
counts = y.value_counts().sort_index()
axes[0].bar(labels, counts.values, color=['steelblue', 'crimson'], edgecolor='black')
axes[0].set_title('Loan Status Distribution', fontsize=13)
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            ax=axes[1], linewidths=0.5, square=True)
axes[1].set_title('Correlation Heatmap', fontsize=13)

df_eda = X_raw.copy()
df_eda['loan_status'] = y.values
df_eda.boxplot(column='credit_score', by='loan_status', ax=axes[2])
axes[2].set_title('Credit Score by Loan Status', fontsize=13)
axes[2].set_xlabel('Loan Status (0=Approved, 1=Default)')
plt.suptitle('')

plt.tight_layout()
plt.show()

## Step 4: Split Data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print('Class distribution in train set:')
print(y_train.value_counts(normalize=True).round(3))

## Step 5: Train Logistic Regression

In [ ]:
model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
model.fit(X_train, y_train)
print('✅ Logistic Regression Model Trained')

## Step 6: Evaluate Model

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(f'Accuracy : {accuracy_score(y_test, y_pred):.4f}')
print(f'ROC AUC  : {roc_auc_score(y_test, y_prob):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Approved', 'Default']))
print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred))

## Step 7: Visualize Results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Approved', 'Default'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix', fontsize=13)

roc_auc = roc_auc_score(y_test, y_prob)
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {roc_auc:.2f}')
axes[1].plot([0,1],[0,1],'--',color='navy',label='Random')
axes[1].set_title('ROC Curve', fontsize=13)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend()

coef = pd.Series(np.abs(model.coef_[0]), index=X.columns).sort_values()
coef.plot(kind='barh', ax=axes[2], color='steelblue')
axes[2].set_title('Feature Importance (|Coefficients|)', fontsize=13)
axes[2].invert_yaxis()

plt.tight_layout()
plt.show()

## Conclusion
- The **real `5_loan_data.csv`** was loaded and preprocessed correctly.
- Missing values handled; categorical columns encoded with **LabelEncoder**; features scaled.
- **EDA** performed: target distribution, correlation heatmap, and credit score boxplot.
- **Logistic Regression** with `class_weight='balanced'` trained on stratified split.
- Model evaluated with Accuracy, Precision, Recall, F1, ROC AUC, and Confusion Matrix.
- Feature coefficients show which factors most influence loan default prediction.